## 1. Set up your environment

This step ensures all the necessary tools are available to build the RAG system. Each library serves a specific role: Langchain handles the orchestration of components, transformers provide pre-trained models, sentence-transformers generate embeddings, datasets load sample data, and FAISS enables fast similarity searches.

In [46]:
# Uninstall existing LangChain packages to ensure a clean install
!pip uninstall -y langchain langchain-community

# Install required libraries
!pip install -Uq langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -Uq langchain-community

Found existing installation: langchain 1.3.11
Uninstalling langchain-1.3.11:
  Successfully uninstalled langchain-1.3.11
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2


## 2. Load the dataset

To provide the system with information to retrieve from, you’ll load a real-world dataset. HuggingFaceDatasetLoader simplifies the process of accessing Hugging Face datasets and formatting them into documents that Langchain can process.

In [41]:
# Install datasets library (if not already installed or to ensure latest version)
!pip install -Uq datasets

# Import HuggingFaceDatasetLoader
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Specify dataset name and content column
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Create HuggingFaceDatasetLoader instance and load data as documents
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Optional: Print the first 2 entries to verify loading
print(data[:2])

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


## 3. Diviser les documents (Split the documents)

Les modèles linguistiques ont une limite à la quantité de texte qu'ils peuvent traiter à la fois. La division de documents volumineux en morceaux plus petits et superposés garantit qu'aucun contexte important n'est perdu et que chaque morceau de texte a une taille gérable pour l'intégration et la récupération.

In [16]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a RecursiveCharacterTextSplitter instance
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Split the loaded documents
docs = text_splitter.split_documents(data)

# Optional: Print the first document chunk
print(docs[0])

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## 4. Intégrer le texte (Embed the text)

Le texte doit être converti en représentations numériques (intégrations) afin que des morceaux de texte similaires puissent être trouvés efficacement. L’utilisation d’un modèle de transformateur de phrases crée des intégrations qui capturent le sens sémantique, permettant une récupération efficace ultérieurement.

In [17]:
# Import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Define model path, model configurations, and encoding options
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialize HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

# (Optional) Creating test embedding:
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[-0.038338519632816315, 0.1234646886587143, -0.028642984107136726]


## 5. Créer un magasin vectoriel (Create a vector store)

Un magasin vectoriel comme FAISS indexe les intégrations, permettant des recherches de similarité rapides et évolutives. C'est ainsi que le système trouve rapidement les morceaux de texte pertinents lorsqu'une requête est effectuée.

In [18]:
# Import FAISS
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store from document chunks and embeddings
db = FAISS.from_documents(docs, embeddings)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


## 6. Prepare the LLM model

The Language Model is responsible for generating answers based on retrieved documents. Loading a pre-trained model and wrapping it in a Langchain pipeline makes it easy to integrate with the retrieval system.

In [59]:
# Install detectron2, which is required for LayoutLMv2 models.
# Installing from source to ensure compatibility in Colab environment.
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

# IMPORTANT: After running this cell, please restart the Colab runtime (Runtime > Restart runtime)
# for detectron2 to be properly recognized, then re-run all cells from the beginning or from cell 816eccc8.

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-62ga3bo2
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-62ga3bo2
  Resolved https://github.com/facebookresearch/detectron2.git to commit 02b5c4e295e990042a714712c21dc79b731e8833
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 12.9 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=6733869 sha2

In [60]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# Load the tokenizer and question-answering model (using a model compatible with document-question-answering)
# Using LayoutLMv2 for document-question-answering as BertForQuestionAnswering is not supported for this task.
tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv2-base-uncased")
model = AutoModelForQuestionAnswering.from_pretrained("microsoft/layoutlmv2-base-uncased")

# Create a document-question-answering pipeline
qa_pipeline = pipeline(
  "document-question-answering",
  model=model, # Pass the loaded model object
  tokenizer=tokenizer
)

# Create a Langchain pipeline wrapper
llm = HuggingFacePipeline(
  pipeline=qa_pipeline,
  model_kwargs={"temperature": 0.7, "max_length": 512},
)

ImportError: 
LayoutLMv2Model requires the detectron2 library but it was not found in your environment. Check out the instructions on the
installation page: https://github.com/facebookresearch/detectron2/blob/master/INSTALL.md and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


## 7. Build the Retrieval QA Chain

The Retrieval QA Chain connects the retriever (which finds relevant documents) with the LLM (which generates answers). This chain enables the full RAG process, where the system retrieves helpful context and then answers the user’s query based on that context.

In [56]:
from langchain.chains import RetrievalQA

# Create a retriever from your FAISS database
retriever = db.as_retriever(search_kwargs={"k": 4}) # Optional: You can adjust k for number of documents retrieved

# Build the RetrievalQA chain
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="refine", retriever=retriever, return_source_documents=False)

print("Retrieval QA Chain built successfully.")

ModuleNotFoundError: No module named 'langchain.chains'

In [49]:
# The user requested to execute cell ddb49267.
# Please manually run cell ddb49267 again to test the import.

## 8. Test your RAG system

Running a test query allows you to verify that all components are working together. This step ensures that documents are retrieved correctly and that the model generates meaningful answers based on the retrieved context.

In [51]:
# Define your question
question = "What is cheesemaking?"

# Run the QA chain and print the result
result = qa.run({"query": question})
print(result)

NameError: name 'qa' is not defined